In [ ]:
# Forcem l'actualitzacio del pip i instal·lem especificant els noms exactes
!pip install --upgrade pip
!pip install segmentation-models-pytorch medmnist

v6 - U-Net Autoencoder prova + Spatial VAE prova - 07022026 

**Simple U-Net autoencoder (encoder ResNet18) amb prova skip dropout 0.5 i noise 0.2**

In [ ]:
# --- 1. CONFIGURACIÓ ---

import torch
from torch import nn, optim
from medmnist import PneumoniaMNIST
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm 
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp 
import gc

torch.cuda.empty_cache()
gc.collect()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usant dispositiu: {DEVICE}")

# --- 2. MODEL: U-Net Autoencoder amb "Skip Dropout" ---

class SimpleUNetAutoencoder(nn.Module):
    def __init__(self, latent_dim=256, skip_dropout_prob=0.5):
        super().__init__()
        
        # 1. Base U-Net (Encoder ResNet18)
        self.base_model = smp.Unet(
            encoder_name="resnet18",        
            encoder_weights=None, 
            in_channels=1,                  
            classes=1,
        )
        
        self.encoder = self.base_model.encoder
        self.decoder = self.base_model.decoder
        self.segmentation_head = self.base_model.segmentation_head
        
        # 2. Configurar "Skip Dropout"
        # Això evita que el model faci trampes usant només les skips.
        self.skip_dropout = nn.Dropout2d(p=skip_dropout_prob)
        
        # 3. Coll d'ampolla (Latent Space Determinista)
        # ResNet18 redueix 224 -> 7x7 amb 512 canals
        self.flat_dim = 512 * 7 * 7 
        
        # Encoder Projection (Imatge -> Vector)
        self.bottleneck_enc = nn.Linear(self.flat_dim, latent_dim)
        
        # Decoder Projection (Vector -> Imatge)
        self.bottleneck_dec = nn.Linear(latent_dim, self.flat_dim)
        
    def forward(self, x):
        # --- A. ENCODER ---
        # Obtenim la llista de features (skips + final)
        features = list(self.encoder(x))
        
        # --- B. CONTROL DE SKIP CONNECTIONS (La part clau) ---
        # Apliquem Dropout a les connexions intermèdies per obligar
        # a passar informació pel centre.
        # features[0] és input, features[-1] és el coll d'ampolla.
        # Les intermèdies són features[1:-1]
        if self.training:
            for i in range(1, len(features)-1):
                features[i] = self.skip_dropout(features[i])

        # --- C. COLL D'AMPOLLA ---
        deepest_map = features[-1] # [B, 512, 7, 7]
        B, C, H, W = deepest_map.shape
        
        # Aplanar i comprimir al vector latent
        flat = deepest_map.view(B, -1)
        z = self.bottleneck_enc(flat) # Aquest és el vector de 256 num!
        
        # Expandir per tornar a pujar
        z_projected = self.bottleneck_dec(z)
        z_map = z_projected.view(B, C, H, W)
        
        # Substituïm el mapa original pel que ve del latent
        features[-1] = z_map
        
        # --- D. DECODER ---
        decoder_output = self.decoder(features)
        reconstruction = self.segmentation_head(decoder_output)
        
        return reconstruction, z

# --- 3. DADES (TURBO DATASET) ---
# Sense canvis
img_size = 224
data_train_raw = PneumoniaMNIST(split="train", download=True, size=img_size)
data_val_raw = PneumoniaMNIST(split="val", download=True, size=img_size)

class TurboDataset(Dataset):
    def __init__(self, data):
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(DEVICE)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx]

train_dataset = TurboDataset(data_train_raw)
val_dataset = TurboDataset(data_val_raw)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# --- 4. ENTRENAMENT (Versió "Hardcore" amb Soroll) ---

# Reiniciem el model amb alta probabilitat de tallar skips
model = SimpleUNetAutoencoder(latent_dim=256, skip_dropout_prob=0.5).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4) 
loss_fn = nn.MSELoss() 
scaler = torch.amp.GradScaler('cuda')

epochs = 40 # Donem-li una mica més de temps perquè ara és més difícil
print("Iniciant entrenament HARDCORE (Soroll + High Dropout)...")

for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    batch_losses = []
    
    for imgs in loop:
        optimizer.zero_grad()
        
        # --- A. AFEGIR SOROLL (Noise Injection) ---
        # Creem soroll gaussià i l'afegim a la imatge
        noise_factor = 0.2
        noisy_imgs = imgs + (noise_factor * torch.randn_like(imgs))
        noisy_imgs = torch.clamp(noisy_imgs, 0., 1.) # Assegurem que segueixi entre 0 i 1
        
        with torch.amp.autocast('cuda'):
            # Li passem la imatge bruta (noisy)
            recon, z = model(noisy_imgs) 
            
            # Però comparem amb la imatge NETA (imgs)
            # Això l'obliga a aprendre a netejar (denoising)
            loss = loss_fn(recon, imgs)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_losses.append(loss.item())
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Loss = {np.mean(batch_losses):.5f}")

# --- 5. VISUALITZACIÓ DEL RESULTAT ---
# Ara tornem a fer la prova de la veritat
print("\n FENT LA PROVA DE LA VERITAT FINAL...")
model.eval()
img_test = val_dataset.images[1].unsqueeze(0)

with torch.no_grad():
    # 1. Pas normal (amb skips obertes al 100% pel eval)
    recon_cheat, _ = model(img_test)
    
    # 2. Pas "Veritat" (tallant manualment les skips)
    features = list(model.encoder(img_test))
    for i in range(1, len(features)-1):
        features[i] = torch.zeros_like(features[i]) # TALL TOTAL
    
    deepest_map = features[-1]
    B, C, H, W = deepest_map.shape
    z = model.bottleneck_enc(deepest_map.view(B, -1))
    z_map = model.bottleneck_dec(z).view(B, C, H, W)
    features[-1] = z_map
    
    decoder_output = model.decoder(features)
    recon_truth = model.segmentation_head(decoder_output)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img_test.cpu().squeeze(), cmap='gray'); ax[0].set_title("Original")
ax[1].imshow(recon_cheat.cpu().squeeze(), cmap='gray'); ax[1].set_title("Reconstrucció (Eval)")
ax[2].imshow(recon_truth.cpu().squeeze(), cmap='gray'); ax[2].set_title("La Veritat (Latent)")
plt.show()

**FASE 1: Forcem el model a aprendre sense les skips. Dropout=1 sense soroll**

In [ ]:
# --- CONFIGURACIÓ FASE 1: BLOQUEIG TOTAL ---
# Posem prob=1.0 per TALLAR totalment les skips. 
# Així forcem que TOTA la informació passi pel vector 'z'.
model = SimpleUNetAutoencoder(latent_dim=256, skip_dropout_prob=1.0).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Pugem una mica el LR (1e-3) perquè aprengui ràpid el gruix
loss_fn = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

# Entrenem menys èpoques, només perquè aprengui l'estructura bàsica
epochs_phase1 = 20 
print("FASE 1: Entrenament sense Skip Connections (Obligant a usar el Latent)...")

for epoch in range(epochs_phase1):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_phase1}", leave=False)
    batch_losses = []
    
    for imgs in loop:
        optimizer.zero_grad()
        
        # NOTA: He tret el soroll afegit (noise_factor) perquè amb Dropout=1.0 ja és prou difícil!
        with torch.amp.autocast('cuda'):
            recon, z = model(imgs) 
            loss = loss_fn(recon, imgs)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_losses.append(loss.item())
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Loss = {np.mean(batch_losses):.5f}")

# Guardem aquest encoder 
torch.save(model.state_dict(), "encoder_phase1_no_skips.pth")

# --- VISUALITZACIÓ FASE 1 ---
print("\nResultat Fase 1 (Sense Skips): Pulmons borrosos però REALS al Latent")
model.eval()
img_test = val_dataset.images[0].unsqueeze(0)

with torch.no_grad():
    # Com que el dropout és 1.0, l'entrenament ja ha simulat la "veritat".
    # Però per visualitzar, fem el tall manual igualment per assegurar-nos.
    features = list(model.encoder(img_test))
    for i in range(1, len(features)-1): features[i] = torch.zeros_like(features[i])
    
    deepest_map = features[-1]
    z = model.bottleneck_enc(deepest_map.view(1, -1))
    z_map = model.bottleneck_dec(z).view(1, 512, 7, 7)
    features[-1] = z_map
     
    recon = model.segmentation_head(model.decoder(features))

plt.imshow(recon.cpu().squeeze(), cmap='gray')
plt.title("Reconstrucció només amb Latent (Fase 1)")
plt.show()

**Fase 2: ràpida (5-10 èpoques) amb skip_dropout=0.5 i LR baix (1e-4) per recuperar la nitidesa HD, carregant els pesos d'aquesta Fase 1.**

In [ ]:
# --- CONFIGURACIÓ FASE 2: REFINAMENT HD ---

# 1. Re-inicialitzem el model, però ara amb Dropout 0.5 (permetem detalls)
model = SimpleUNetAutoencoder(latent_dim=256, skip_dropout_prob=0.5).to(DEVICE)

# 2. Carreguem els pesos de la Fase 1 (l'estructura apresa)
try:
    model.load_state_dict(torch.load("encoder_phase1_no_skips.pth"))
    print(" Pesos de la Fase 1 carregats correctament! entrenant...")
except FileNotFoundError:
    print("ALERTA: No trobo l'arxiu 'encoder_phase1_no_skips.pth'. Assegura't d'haver executat la Fase 1.")

# 3. Learning Rate més baix (Fine-Tuning)
# Baixem de 1e-3 a 1e-4 per no destruir l'estructura latent, només afegir detalls.
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

# Fase 2 sembla ràpida, amb 10-15 èpoques per ara n'hi ha prou
epochs_phase2 = 15
print("FASE 2: Recuperant detalls HD (Skip Dropout = 0.5)...")

for epoch in range(epochs_phase2):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_phase2}", leave=False)
    batch_losses = []
    
    for imgs in loop:
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            # Ara ja no cal fer res especial, el dropout de 0.5 actua sol
            recon, z = model(imgs)
            loss = loss_fn(recon, imgs)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_losses.append(loss.item())
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Loss = {np.mean(batch_losses):.5f}")

# Guardem el model definitiu
torch.save(model.state_dict(), "unet_autoencoder_final_HD.pth")

# --- VISUALITZACIÓ FINAL ---
print("\n Resultat Final (Fase 2): Hauria de ser NÍTID i detallat")
model.eval()
img_test = val_dataset.images[0].unsqueeze(0)

with torch.no_grad():
    # Reconstrucció normal (aprofitant les Skips com farà en producció)
    recon_hd, _ = model(img_test)
    
    # Per curiositat, mirem què passa si tallem els cables ara
    # (Hauria de mantenir l'estructura de la Fase 1, encara que perdi qualitat)
    features = list(model.encoder(img_test))
    for i in range(1, len(features)-1): features[i] = torch.zeros_like(features[i])
    deepest_map = features[-1]
    z = model.bottleneck_enc(deepest_map.view(1, -1))
    z_map = model.bottleneck_dec(z).view(1, 512, 7, 7)
    features[-1] = z_map
    recon_latent = model.segmentation_head(model.decoder(features))

# Comparativa
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_test.cpu().squeeze(), cmap='gray'); ax[0].set_title("Original")
ax[1].imshow(recon_hd.cpu().squeeze(), cmap='gray'); ax[1].set_title("Reconstrucció HD (Fase 2)")
ax[2].imshow(recon_latent.cpu().squeeze(), cmap='gray'); ax[2].set_title("Només Latent (Estructura)")
plt.show()

**Spatial VAE (Arquitectura State of the Art)**

In [ ]:
# --- 1. CONFIGURACIÓ I LLIBRERIES ---
import torch
from torch import nn, optim
from medmnist import PneumoniaMNIST
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import gc

# Neteja de memòria per si de cas
torch.cuda.empty_cache()
gc.collect()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Usant dispositiu: {DEVICE}")

# --- 2. EL MODEL: VAE U-Net Híbrid (Segmentation Models PyTorch + Latent Probabilístic) ---

class VAE_UNet(nn.Module):
    def __init__(self, latent_dim=256, skip_dropout_prob=0.5):
        super().__init__()
        
        # 1. Base U-Net (ResNet18) de la llibreria
        self.base_model = smp.Unet(encoder_name="resnet18", encoder_weights=None, in_channels=1, classes=1)
        
        # Desempaquetem les peces
        self.encoder = self.base_model.encoder
        self.decoder = self.base_model.decoder
        self.segmentation_head = self.base_model.segmentation_head
        
        # 2. Control de Skip Connections (Vital per no fer trampes)
        self.skip_dropout = nn.Dropout2d(p=skip_dropout_prob)
        
        # 3. Coll d'Ampolla VAE
        # ResNet18 acaba amb 512 canals i mida 7x7 (si entra 224x224)
        self.flat_dim = 512 * 7 * 7 
        
        # Projeccions a Mitjana (Mu) i Log-Variància (LogVar)
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_var = nn.Linear(self.flat_dim, latent_dim)
        
        # Projecció de tornada (Decoder Input)
        self.decoder_input = nn.Linear(latent_dim, self.flat_dim)
        
    def reparameterize(self, mu, log_var):
        """ VAE: Samplejar amb soroll controlat"""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        # --- A. Encoder ---
        features = list(self.encoder(x))
        
        # --- B. Skip Dropout (Entrenament) ---
        if self.training:
            for i in range(1, len(features)-1):
                features[i] = self.skip_dropout(features[i])

        # --- C. VAE Bottleneck ---
        deepest_map = features[-1] 
        B, C, H, W = deepest_map.shape
        flat = deepest_map.view(B, -1)
        
        # Calculem Mu i LogVar
        mu = self.fc_mu(flat)
        log_var = self.fc_var(flat)
        
        # Si entrenem -> Samplejem (afegim aleatorietat)
        # Si avaluem -> Usem la mitjana directa (imatge neta)
        if self.training:
            z = self.reparameterize(mu, log_var)
        else:
            z = mu 
            
        # --- D. Decoder ---
        # Projectem z de nou a mapa de característiques
        z_projected = self.decoder_input(z)
        z_map = z_projected.view(B, C, H, W)
        
        # REEMPLACEM el mapa original pel nostre mapa generat des del latent
        features[-1] = z_map 
        
        # Passem pel decoder 
        decoder_output = self.decoder(features) 
        reconstruction = self.segmentation_head(decoder_output)
        
        return reconstruction, mu, log_var

# --- 3. LOSS FUNCTION (VAE Loss) ---
def vae_loss_fn(recon_x, x, mu, log_var, beta=0.00001):
    # beta molt petita per prioritzar qualitat visual (reconstrucció)
    recon_loss = nn.MSELoss()(recon_x, x)
    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + (beta * kl_loss), recon_loss, kl_loss

# --- 4. PREPARACIÓ DE DADES ---
img_size = 224
data_train_raw = PneumoniaMNIST(split="train", download=True, size=img_size)
data_val_raw = PneumoniaMNIST(split="val", download=True, size=img_size)

class TurboDataset(Dataset):
    def __init__(self, data):
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(DEVICE)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx]

train_loader = DataLoader(TurboDataset(data_train_raw), batch_size=64, shuffle=True)
val_dataset = TurboDataset(data_val_raw)

# --- 5. ENTRENAMENT DEL VAE ---
# Creem el model VAE nou
model = VAE_UNet(latent_dim=256, skip_dropout_prob=0.5).to(DEVICE)

# Si volem aprofitar la Fase 2, podem carregar parcialment els pesos.

try:
    state_dict = torch.load("unet_autoencoder_final.pth")
    # Filtrem les claus que no coincideixen (perquè hem afegit mu/var)
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    print("Pesos de la U-Net (Encoder/Decoder) carregats!")
except:
    print("Entrenant VAE des de zero (és ràpid).")

optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda')

epochs = 30 
print("Iniciant entrenament VAE FINAL (amb Interpolació)...")

for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    total_recon = 0
    total_kl = 0
    
    for imgs in loop:
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            recon, mu, log_var = model(imgs)
            # La beta controla l'equilibri. Si surt molt borrós, baixem la beta.
            loss, r_loss, k_loss = vae_loss_fn(recon, imgs, mu, log_var, beta=0.00001)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_recon += r_loss.item()
        total_kl += k_loss.item()
        loop.set_postfix(recon=r_loss.item(), kl=k_loss.item())
        
    print(f"Epoch {epoch+1}: Recon Loss = {total_recon/len(train_loader):.5f}")

# Guardem el model definitiu
torch.save(model.state_dict(), "vae_unet_completed.pth")

# --- 6. GENERAR INTERPOLACIÓ ---

print("\n GENERANT INTERPOLACIÓ REAL (Sense interferència de Skips)...")
model.eval()

# 1. Assegurem-nos que A i B són diferents
img_A = val_dataset.images[0].unsqueeze(0) 
img_B = val_dataset.images[5].unsqueeze(0) 

# Comprovació visual ràpida
if torch.equal(img_A, img_B):
    print("ALERTA: Les imatges A i B són idèntiques! Canvio l'índex B...")
    img_B = val_dataset.images[10].unsqueeze(0)

with torch.no_grad():
    # 2. Obtenim els vectors 'mu' purs
    _, mu_A, _ = model(img_A)
    _, mu_B, _ = model(img_B)
    
    # 3. Seqüència d'interpolació
    alphas = [0, 0.25, 0.5, 0.75, 1.0]
    interpolated_imgs = []
    
    for alpha in alphas:
        # Barreja lineal dels vectors (Latent Space)
        z_inter = (1 - alpha) * mu_A + alpha * mu_B
        
        # --- EL CANVI CLAU ÉS AQUÍ ---
        # Agafem les features de A només per tenir la estructura de llista
        features_dummy = list(model.encoder(img_A))
        
        # TALL RADICAL: Posem a ZERO totes les skips intermèdies
        # Així evitem que el "cos" del Pacient A tapi la interpolació
        for i in range(1, len(features_dummy)-1):
            features_dummy[i] = torch.zeros_like(features_dummy[i])
            
        # Inserim el latent híbrid al final
        z_proj = model.decoder_input(z_inter).view(1, 512, 7, 7)
        features_dummy[-1] = z_proj 
        
        # Decodifiquem (ara el decoder només té el latent per treballar)
        recon_inter = model.segmentation_head(model.decoder(features_dummy))
        interpolated_imgs.append(recon_inter)

# Visualitzem
fig, ax = plt.subplots(1, 7, figsize=(20, 6))

# Mostrem l'Original A i B als extrems per comparar
ax[0].imshow(img_A.cpu().squeeze(), cmap='gray'); ax[0].set_title("Original A")
ax[6].imshow(img_B.cpu().squeeze(), cmap='gray'); ax[6].set_title("Original B")

# Mostrem la interpolació al mig (0% a 100% del latent)
titles = ["0% (Latent A)", "25%", "50% (Híbrid)", "75%", "100% (Latent B)"]
for i in range(5):
    ax[i+1].imshow(interpolated_imgs[i].cpu().squeeze(), cmap='gray')
    ax[i+1].set_title(titles[i])
    # Traiem eixos per neteja
    ax[i+1].axis('off')

ax[0].axis('off'); ax[6].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Redefinim la classe per incloure les ETIQUETES (labels)
class TurboDataset(Dataset):
    def __init__(self, data):
        # Imatges
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(DEVICE)
        # Etiquetes (ARA SÍ!)
        self.labels = torch.tensor(data.labels, dtype=torch.long).to(DEVICE)
        
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

# 2. Tornem a carregar el dataset de validació amb la nova classe
# (No cal tornar a descarregar, usum 'data_val_raw' que ja tenim en memòria)
val_dataset = TurboDataset(data_val_raw)

# 3. Ara sí, mirem qui és qui
idx_A = 0
idx_B = 5 

# Recuperem l'etiqueta (fem .squeeze() per treure dimensions extra)
label_A = val_dataset.labels[idx_A].squeeze().item()
label_B = val_dataset.labels[idx_B].squeeze().item()

class_names = {0: " SA (Normal)", 1: " PNEUMÒNIA"}

print(f"Pacient A : {class_names[label_A]}")
print(f"Pacient B :    {class_names[label_B]}")


In [ ]:
# --- BUSQUEM UN PARELL SA vs MALALT ---
print("Buscant un pacient SA i un amb PNEUMÒNIA...")

idx_healthy = -1
idx_pneumonia = -1

# Recorrem el dataset fins trobar un de cada
for i in range(len(val_dataset)):
    label = val_dataset.labels[i].item()
    if label == 0 and idx_healthy == -1: idx_healthy = i
    if label == 1 and idx_pneumonia == -1: idx_pneumonia = i
    if idx_healthy != -1 and idx_pneumonia != -1: break

print(f"Trobat SA: índex {idx_healthy}")
print(f"Trobat PNEUMÒNIA: índex {idx_pneumonia}")

# --- FEM LA INTERPOLACIÓ LINEAL ENTRE ELS DOS ---
model.eval()
img_healthy = val_dataset.images[idx_healthy].unsqueeze(0)
img_sick = val_dataset.images[idx_pneumonia].unsqueeze(0)

with torch.no_grad():
    _, mu_H, _ = model(img_healthy) # Vector del Sa
    _, mu_S, _ = model(img_sick)    # Vector del Malalt
    
    alphas = [0, 0.25, 0.5, 0.75, 1.0]
    interpolated_imgs = []
    
    for alpha in alphas:
        # Viatgem del Sa (0) al Malalt (1)
        z_inter = (1 - alpha) * mu_H + alpha * mu_S
        
        # Sense skips
        features_dummy = list(model.encoder(img_healthy))
        for i in range(1, len(features_dummy)-1): 
            features_dummy[i] = torch.zeros_like(features_dummy[i])
            
        z_proj = model.decoder_input(z_inter).view(1, 512, 7, 7)
        features_dummy[-1] = z_proj 
        
        recon = model.segmentation_head(model.decoder(features_dummy))
        interpolated_imgs.append(recon)

# --- VISUALITZEM LA "INFECCIÓ" ---
fig, ax = plt.subplots(1, 5, figsize=(20, 5))
titles = ["0% (SA - Pulmó Net)", "25%", "50%", "75%", "100% (PNEUMÒNIA - Pulmó Blanc)"]

for i in range(5):
    ax[i].imshow(interpolated_imgs[i].cpu().squeeze(), cmap='gray')
    ax[i].set_title(titles[i])
    ax[i].axis('off')

plt.suptitle("SIMULACIÓ D'APARICIÓ DE PNEUMÒNIA (Latent)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# --- DIAGNÒSTIC: SÓN IGUALS DE VERITAT? ---

# 1. Recuperem els vectors latents del Sa (H) i del Malalt (S) que hem usat abans
# (Assumeixo que les variables img_healthy i img_sick encara existeixen de la cel·la anterior)
with torch.no_grad():
    _, mu_H, _ = model(img_healthy)
    _, mu_S, _ = model(img_sick)
    
    # 2. Calculem la diferència numèrica entre els vectors latents
    distancia = torch.norm(mu_H - mu_S).item()
    print(f"Distància Euclidiana entre vectors Latents: {distancia:.4f}")
    
    if distancia < 1.0:
        print("ALERTA: Els vectors són massa propers! El model no distingeix bé aquests dos pacients.")
    else:
        print("ELS VECTORS SÓN DIFERENTS. El canvi existeix, encara que costi de veure.")

# --- VISUALITZACIÓ DE LA DIFERÈNCIA (HEATMAP) ---
# En lloc de veure grisos, veurem la RESTA.
# Vermell = Aquí ha aparegut teixit (Pneumònia/Inflamació)
# Blau = Aquí ha desaparegut teixit

from matplotlib.colors import TwoSlopeNorm

# Generem les imatges al latent
with torch.no_grad():
    # Sa
    feats_H = list(model.encoder(img_healthy))
    for i in range(1, len(feats_H)-1): feats_H[i] = torch.zeros_like(feats_H[i])
    feats_H[-1] = model.decoder_input(mu_H).view(1, 512, 7, 7)
    recon_H = model.segmentation_head(model.decoder(feats_H))
    
    # Malalt
    feats_S = list(model.encoder(img_sick))
    for i in range(1, len(feats_S)-1): feats_S[i] = torch.zeros_like(feats_S[i])
    feats_S[-1] = model.decoder_input(mu_S).view(1, 512, 7, 7)
    recon_S = model.segmentation_head(model.decoder(feats_S))

# Calculem la diferència
diff = (recon_S - recon_H).cpu().squeeze().numpy()

plt.figure(figsize=(10, 8))
# Usem un mapa de colors que ressalti els extrems (bwr = Blue-White-Red)
# Centrem el blanc al 0 (sense canvi)
norm = TwoSlopeNorm(vmin=diff.min(), vcenter=0, vmax=diff.max())
plt.imshow(diff, cmap='bwr', norm=norm)
plt.colorbar(label="Diferència de Densitat")
plt.title(f"MAPA DE CANVIS: On està la Pneumònia?\n(Vermell = Més densitat/Infecció)", fontsize=14)
plt.axis('off')
plt.show()

print(f"Max diferència positiva (Vermell): {diff.max():.4f}")
print(f"Max diferència negativa (Blau):    {diff.min():.4f}")

**Extraccio de Latents**

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# --- 1. CONFIGURACIÓ ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
IMG_SIZE = 224

# Assegurar que tenim la classe VAE_UNet definida o importada!
# (Assumim en aquesta part del codi que el model ja està carregat a la variable 'model' de l'entrenament anterior.
#  Si no, caldrà descomentar les línies de sota per carregar-lo de disc)

# model = VAE_UNet(latent_dim=256, skip_dropout_prob=0.0).to(DEVICE) # Dropout 0 per extracció!
# model.load_state_dict(torch.load("vae_unet_completed.pth"))

model.eval() #  Mode avaluació (apaga Dropouts)

# --- 2. PREPARAR DATASETS ---
# Volem extreure latents tant de TRAIN (per entrenar el Flow) com de VAL/TEST
from medmnist import PneumoniaMNIST
from torch.utils.data import Dataset

class TurboDataset(Dataset):
    def __init__(self, data):
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(DEVICE)
        # Guardem també les etiquetes (labels) per si volem fer Flow Condicional (Pneumonia vs Sa)
        self.labels = torch.tensor(data.labels, dtype=torch.long).to(DEVICE)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

data_train_raw = PneumoniaMNIST(split="train", download=True, size=IMG_SIZE)
data_val_raw = PneumoniaMNIST(split="val", download=True, size=IMG_SIZE)
data_test_raw = PneumoniaMNIST(split="test", download=True, size=IMG_SIZE)

loaders = {
    'train': DataLoader(TurboDataset(data_train_raw), batch_size=BATCH_SIZE, shuffle=False),
    'val': DataLoader(TurboDataset(data_val_raw), batch_size=BATCH_SIZE, shuffle=False),
    'test': DataLoader(TurboDataset(data_test_raw), batch_size=BATCH_SIZE, shuffle=False)
}

# --- 3. BUCLE D'EXTRACCIÓ ---
print("Iniciant extracció de vectors latents...")

for split, loader in loaders.items():
    latents_list = []
    labels_list = []
    
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc=f"Extracting {split}"):
            # Passem pel model
            # El model retorna: reconstruction, mu, log_var
            # NOMÉS ENS INTERESSA 'mu' (la mitjana és el punt més representatiu)
            _, mu, _ = model(imgs)
            
            latents_list.append(mu.cpu().numpy())
            labels_list.append(lbls.cpu().numpy())
    
    # Concatenem tot
    latents_array = np.concatenate(latents_list, axis=0)
    labels_array = np.concatenate(labels_list, axis=0)
    
    print(f" {split.upper()}: Guardat array de forma {latents_array.shape}")
    
    # Guardem a disc
    np.save(f"latents_{split}.npy", latents_array)
    np.save(f"labels_{split}.npy", labels_array)

print("\n Tenim 6 arxius nous (.npy) preparats per al Flow Matching.")
print("  - latents_train.npy és el més important.")

In [ ]:
from sklearn.decomposition import PCA

print("Dibuixant el mapa de l'espai latent...")

# 1. Agafem uns quants pacients (ex: 500) per fer de "fons" del mapa
latents_map = []
labels_map = []
num_samples = 500

with torch.no_grad():
    for i in range(num_samples):
        img = val_dataset.images[i].unsqueeze(0)
        lbl = val_dataset.labels[i].item()
        _, mu, _ = model(img)
        latents_map.append(mu.cpu().numpy().flatten())
        labels_map.append(lbl)

latents_map = np.array(latents_map)
labels_map = np.array(labels_map)

# 2. Agafem la nostra trajectòria A -> B
trajectory_points = []
alphas = np.linspace(0, 1, 10) # 10 punts al llarg del camí

with torch.no_grad():
    _, mu_A, _ = model(img_A)
    _, mu_B, _ = model(img_B)
    
    for alpha in alphas:
        z_inter = (1 - alpha) * mu_A + alpha * mu_B
        trajectory_points.append(z_inter.cpu().numpy().flatten())

trajectory_points = np.array(trajectory_points)

# 3. COMPRIMIM tot a 2D usant PCA
# Entrenem el PCA amb tot junt perquè entengui l'espai
all_data = np.concatenate([latents_map, trajectory_points], axis=0)
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_data)

# Separem un altre cop
map_2d = all_2d[:num_samples]
traj_2d = all_2d[num_samples:]

# 4. PINTEM EL GRÀFIC
plt.figure(figsize=(10, 8))

# Fons: Pacients Normals vs Pneumònia
plt.scatter(map_2d[labels_map==0, 0], map_2d[labels_map==0, 1], c='blue', alpha=0.3, label='Sans', s=10)
plt.scatter(map_2d[labels_map==1, 0], map_2d[labels_map==1, 1], c='red', alpha=0.3, label='Pneumònia', s=10)

# La Trajectòria
plt.plot(traj_2d[:, 0], traj_2d[:, 1], 'k--', linewidth=2, label='Trajectòria Interpolada')

# Inici i Fi
plt.scatter(traj_2d[0, 0], traj_2d[0, 1], c='lime', s=200, marker='*', edgecolors='black', label='Inici (A)', zorder=5)
plt.scatter(traj_2d[-1, 0], traj_2d[-1, 1], c='yellow', s=200, marker='*', edgecolors='black', label='Fi (B)', zorder=5)

plt.title(f"Visualització de la Trajectòria a l'Espai Latent (256D -> 2D)\nDistància Real: {torch.norm(mu_A - mu_B):.2f}", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel("Component Principal 1")
plt.ylabel("Component Principal 2")
plt.show()

**Versio 2 Spatial VAE Antiexplosions d'exponencials**

In [ ]:
# Forcem l'actualització del pip i instal·lem especificant els noms exactes
!pip install --upgrade pip
!pip install segmentation-models-pytorch medmnist

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import segmentation_models_pytorch as smp
from medmnist import PneumoniaMNIST
from sklearn.decomposition import PCA
import os
import gc

# --- 1. CONFIGURACIÓ SEGURA (SAFE MODE) ---
CONFIG = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 64,      # A Kaggle es pots provar 128 si es vol anar ràpid
    "LATENT_CHANNELS": 4,  
    "EPOCHS_PHASE_1": 10,  # Reduït una mica per proves ràpides
    "EPOCHS_PHASE_2": 10,
    "LR_PHASE_1": 1e-4,    # MOLT IMPORTANT: Més lent per no explotar (era 1e-3)
    "LR_PHASE_2": 1e-5,    # Refinament molt suau
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "SAVE_DIR": "./artifacts"
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
print(f" Iniciant Pipeline al dispositiu: {CONFIG['DEVICE']}")

# --- 2. DATASET ---
class TurboDataset(Dataset):
    def __init__(self, data):
        # Normalitzem entre 0 i 1
        self.images = torch.tensor(data.imgs, dtype=torch.uint8).unsqueeze(1).float().div(255.0).to(CONFIG["DEVICE"])
        self.labels = torch.tensor(data.labels, dtype=torch.long).to(CONFIG["DEVICE"])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def get_dataloaders():
    data_train = PneumoniaMNIST(split="train", download=True, size=CONFIG["IMG_SIZE"])
    data_val = PneumoniaMNIST(split="val", download=True, size=CONFIG["IMG_SIZE"])
    loaders = {
        'train': DataLoader(TurboDataset(data_train), batch_size=CONFIG["BATCH_SIZE"], shuffle=True),
        'val':   DataLoader(TurboDataset(data_val), batch_size=CONFIG["BATCH_SIZE"], shuffle=False),
        'val_ds': TurboDataset(data_val) 
    }
    return loaders

# --- 3. MODEL: SPATIAL VAE (ANTI-EXPLOSIÓ) ---
class SpatialVAE_UNet(nn.Module):
    def __init__(self, latent_channels=4, skip_dropout_prob=0.5):
        super().__init__()
        self.base_model = smp.Unet(encoder_name="resnet18", in_channels=1, classes=1)
        self.encoder = self.base_model.encoder
        self.decoder = self.base_model.decoder
        self.segmentation_head = self.base_model.segmentation_head
        self.skip_dropout = nn.Dropout2d(p=skip_dropout_prob)
        
        # Coll d'ampolla espacial
        self.bottleneck_conv = nn.Conv2d(512, 2 * latent_channels, kernel_size=1)
        self.decoder_input_conv = nn.Conv2d(latent_channels, 512, kernel_size=1)
        
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        features = list(self.encoder(x))
        if self.training:
            for i in range(1, len(features)-1):
                features[i] = self.skip_dropout(features[i])

        deepest_map = features[-1] 
        latent_dist = self.bottleneck_conv(deepest_map)
        mu, log_var = torch.chunk(latent_dist, 2, dim=1)
        
        # PROTECCIÓ CLAU: Clamping del Log-Var
        # Evita que l'exponencial exploti a Infinit
        log_var = torch.clamp(log_var, min=-10, max=10)
        
        if self.training:
            z = self.reparameterize(mu, log_var)
        else:
            z = mu 
            
        z_expanded = self.decoder_input_conv(z)
        features[-1] = z_expanded
        
        decoder_output = self.decoder(features) 
        reconstruction = self.segmentation_head(decoder_output)
        
        return reconstruction, mu, log_var

def vae_loss_fn(recon_x, x, mu, log_var, beta=0.00001):
    recon_loss = nn.MSELoss()(recon_x, x)
    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
    # Protecció contra NaN a la Loss
    if torch.isnan(kl_loss): kl_loss = torch.tensor(0.0, device=x.device)
    return recon_loss + (beta * kl_loss), recon_loss, kl_loss

# --- 4. ENGINE D'ENTRENAMENT (ROBUST) ---
def train_phase(model, loader, optimizer, epochs, phase_name):
    model.train()
    print(f"\n INICIANT {phase_name} | Dropout: {model.skip_dropout.p} | LR: {optimizer.param_groups[0]['lr']}")
    
    for epoch in range(epochs):
        loop = tqdm(loader, desc=f"Ep {epoch+1}/{epochs}", leave=False)
        epoch_recon = 0
        
        for imgs, _ in loop:
            optimizer.zero_grad()
            
            # Forward (sense AMP Automatic Mixed Precision treballem amb float32 per seguretat)
            recon, mu, log_var = model(imgs)
            
            # Comprovació d'emergència
            if torch.isnan(recon).any():
                print(" ALERTA: NaN detectat a la sortida del model! Saltant batch...")
                continue

            loss, r_loss, _ = vae_loss_fn(recon, imgs, mu, log_var)
            
            if torch.isnan(loss):
                print(" ALERTA: Loss és NaN! Saltant update...")
                continue
                
            loss.backward()
            
            #  PROTECCIÓ: Gradient Clipping (Talla els gradients gegants)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_recon += r_loss.item()
            loop.set_postfix(loss=f"{r_loss.item():.4f}")
            
        avg_loss = epoch_recon/len(loader)
        print(f"   -> Ep {epoch+1} Avg MSE: {avg_loss:.5f}")
        
        if np.isnan(avg_loss):
            print(" ERROR FATAL: El model ha col·lapsat a NaN. Aturant entrenament.")
            return

# --- 5. VISUALITZACIÓ SEGURA ---
def visualize_safe(model, val_ds):
    model.eval()
    print("\n Generant resultats visuals...")
    
    # Busquem índexs
    idx_H, idx_P = 0, 1
    for i in range(len(val_ds)):
        if val_ds.labels[i].item() == 0: idx_H = i
        if val_ds.labels[i].item() == 1: idx_P = i; break
            
    img_H = val_ds.images[idx_H].unsqueeze(0)
    img_P = val_ds.images[idx_P].unsqueeze(0)
    
    with torch.no_grad():
        rec_H, _, _ = model(img_H)
        rec_P, _, _ = model(img_P)
        
        # Correcció de rang (Sigmoid si cal)
        if rec_H.min() < 0 or rec_H.max() > 1.5:
            rec_H = torch.sigmoid(rec_H)
            rec_P = torch.sigmoid(rec_P)
            print("   -> Aplicat Sigmoid per visualitzar.")

    # Plot Reconstrucció
    fig, ax = plt.subplots(2, 2, figsize=(8, 8))
    ax[0,0].imshow(img_H.cpu().squeeze(), cmap='gray'); ax[0,0].set_title("Original SA")
    ax[0,1].imshow(rec_H.cpu().squeeze(), cmap='gray'); ax[0,1].set_title("Reconstrucció SA")
    ax[1,0].imshow(img_P.cpu().squeeze(), cmap='gray'); ax[1,0].set_title("Original PNEUMÒNIA")
    ax[1,1].imshow(rec_P.cpu().squeeze(), cmap='gray'); ax[1,1].set_title("Reconstrucció PNEUMÒNIA")
    plt.savefig(f"{CONFIG['SAVE_DIR']}/reconstruccio_final.png")
    plt.show()

# --- 6. EXTRACCIÓ DE LATENTS ---
def extract_latents(model, loaders):
    print("\n Guardant Latents...")
    model.eval()
    for split in ['train', 'val']:
        latents, labels = [], []
        with torch.no_grad():
            for imgs, lbls in tqdm(loaders[split], desc=split):
                _, mu, _ = model(imgs)
                latents.append(mu.cpu().numpy())
                labels.append(lbls.cpu().numpy())
        np.save(f"{CONFIG['SAVE_DIR']}/spatial_latents_{split}.npy", np.concatenate(latents))
        np.save(f"{CONFIG['SAVE_DIR']}/labels_{split}.npy", np.concatenate(labels))
    print(" Tot guardat!")

# --- MAIN ---
if __name__ == "__main__":
    # Neteja de memòria
    gc.collect()
    torch.cuda.empty_cache()
    
    loaders = get_dataloaders()
    model = SpatialVAE_UNet(latent_channels=CONFIG["LATENT_CHANNELS"]).to(CONFIG["DEVICE"])
    
    # Fase 1
    model.skip_dropout.p = 1.0
    opt_1 = optim.AdamW(model.parameters(), lr=CONFIG["LR_PHASE_1"])
    train_phase(model, loaders['train'], opt_1, CONFIG["EPOCHS_PHASE_1"], "FASE 1")
    
    # Fase 2
    model.skip_dropout.p = 0.5
    opt_2 = optim.AdamW(model.parameters(), lr=CONFIG["LR_PHASE_2"])
    train_phase(model, loaders['train'], opt_2, CONFIG["EPOCHS_PHASE_2"], "FASE 2")
    
    # Resultats
    torch.save(model.state_dict(), f"{CONFIG['SAVE_DIR']}/spatial_vae_safe.pth")
    visualize_safe(model, loaders['val_ds'])
    extract_latents(model, loaders)

Gràfics: Iterpol·lació i PCA amb distàncies

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import torch

def generate_plots(model, loader):
    model.eval()
    print("Generant gràfics (Amb Distància Numèrica)...")
    
    # --- GRÀFIC 1: INTERPOLACIÓ REAL (Transició Sa -> Malalt) ---
    iterator = iter(loader)
    imgs, labels = next(iterator)
    imgs = imgs.to(CONFIG["DEVICE"])
    
    # Busquem índexs
    idx_H = (labels == 0).nonzero(as_tuple=True)[0][0]
    idx_P = (labels == 1).nonzero(as_tuple=True)[0][0]
    
    img_H = imgs[idx_H].unsqueeze(0)
    img_P = imgs[idx_P].unsqueeze(0)
    
    alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
    titles = ["Sa (Original)", "25% Pneumònia", "50% Híbrid", "75% Pneumònia", "Pneumònia (Original)"]
    
    fig, ax = plt.subplots(1, 5, figsize=(15, 4))
    
    with torch.no_grad():
        _, mu_H, _ = model(img_H)
        _, mu_P, _ = model(img_P)
        
        for i, alpha in enumerate(alphas):
            z_inter = (1 - alpha) * mu_H + alpha * mu_P
            z_expanded = model.decoder_input_conv(z_inter)
            
            # Interpolació Estructural (Sense Skips)
            feats = list(model.encoder(img_H)) 
            for k in range(len(feats)-1): feats[k] = torch.zeros_like(feats[k])
            feats[-1] = z_expanded
            
            recon = model.segmentation_head(model.decoder(feats))
            
            # Visualització neta
            img_vis = recon.cpu().squeeze().numpy()
            if img_vis.max() > 1.5 or img_vis.min() < 0:
                img_vis = 1 / (1 + np.exp(-img_vis))
                
            ax[i].imshow(img_vis, cmap='gray', vmin=0, vmax=1)
            ax[i].set_title(titles[i], fontsize=10)
            ax[i].axis('off')
            
    plt.tight_layout()
    plt.savefig(f"{CONFIG['SAVE_DIR']}/figura_interpolacio.png", dpi=300)
    plt.show()

    # --- GRÀFIC 2: MAPA PCA AMB DISTÀNCIA NUMÈRICA  ---
    latents, labels_list = [], []
    max_samples = 500
    
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(CONFIG["DEVICE"])
            _, mu, _ = model(imgs)
            latents.append(mu.view(mu.size(0), -1).cpu().numpy())
            labels_list.append(lbls.cpu().numpy())
            if len(latents) * CONFIG["BATCH_SIZE"] >= max_samples: break
            
    X = np.concatenate(latents, axis=0)
    y = np.concatenate(labels_list, axis=0).squeeze()
    
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)
    
    # Centres
    center_H = X_2d[y==0].mean(axis=0)
    center_P = X_2d[y==1].mean(axis=0)
    
    # CÀLCUL DE LA DISTÀNCIA
    dist = np.linalg.norm(center_H - center_P)
    
    plt.figure(figsize=(10, 8))
    # Punts
    plt.scatter(X_2d[y==0, 0], X_2d[y==0, 1], c='#1f77b4', alpha=0.5, label='Classe: NORMAL')
    plt.scatter(X_2d[y==1, 0], X_2d[y==1, 1], c='#d62728', alpha=0.5, label='Classe: PNEUMONIA')
    
    # Fletxa de transició
    plt.annotate("", xy=center_P, xytext=center_H,
                 arrowprops=dict(arrowstyle="->", lw=3, color='black'))
    
    # ETIQUETA AMB EL NÚMERO
    mid_x = (center_H[0] + center_P[0]) / 2
    mid_y = (center_H[1] + center_P[1]) / 2
    
    plt.text(mid_x, mid_y, f"Distància: {dist:.2f}", 
             ha='center', va='center', fontsize=12, fontweight='bold',
             bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Marcadors
    plt.scatter(*center_H, c='navy', s=200, marker='o', edgecolors='white', label='Centroide SA')
    plt.scatter(*center_P, c='darkred', s=200, marker='X', edgecolors='white', label='Centroide PNEUMONIA')
    
    plt.legend()
    plt.title(f"Espai Latent PCA (Distància = {dist:.2f})", fontsize=14)
    plt.xlabel("Component Principal 1")
    plt.ylabel("Component Principal 2")
    plt.grid(True, alpha=0.3)
    
    plt.savefig(f"{CONFIG['SAVE_DIR']}/figura_pca.png", dpi=300)
    plt.show()
    print(f"Gràfic PCA generat amb distància: {dist:.4f}")

# Executar
generate_plots(model, loaders['val'])

Espresso Macchiato porfavore...